# 🏦 Retail Banking Product & Fees Assistant
### Hybrid RAG with Intent Routing, Grounded Citations & Refusal Mechanism

**Project:** Capstone — GenAI Banking Assistant  
**Stack:** FAISS + BM25 + RRF | Groq LLM | LLM-as-Judge Evaluator

---
**Pipeline Overview:**
```
Query → Intent Router → Filtered Retrieval (Dense + Sparse + RRF)
      → Confidence Check → Generator (with citations) → LLM-as-Judge Eval
```

## 📦 Section 1: Install Dependencies

In [ ]:
!pip install -q faiss-cpu rank_bm25 sentence-transformers groq langchain langchain-community langchain-text-splitters tiktoken


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 🔑 Section 2: API Key Setup

In [1]:
import os
# from google.colab import userdata

# Store your key in Colab Secrets (key icon in left sidebar) as GROQ_API_KEY
# os.environ["GROQ_API_KEY"] = "your-groq-api-key-here"  # Set your key in Colab Secrets as GROQ_API_KEY

from groq import Groq
client = Groq(api_key=os.environ["GROQ_API_KEY"])
print("✅ Groq client initialized")

import time as _time
from functools import wraps
from groq import RateLimitError

# Wrap client.chat.completions.create to retry on 429 rate limits
_orig_create = client.chat.completions.create

@wraps(_orig_create)
def _create_with_retry(*args, **kwargs):
    max_retries = 8
    backoff = 1.5
    for attempt in range(max_retries):
        try:
            return _orig_create(*args, **kwargs)
        except RateLimitError as e:
            if attempt == max_retries - 1:
                raise
            wait = backoff * (attempt + 1)
            print(f"⚠️ Rate limit hit, retrying in {wait:.1f}s (attempt {attempt+2}/{max_retries})")
            _time.sleep(wait)

client.chat.completions.create = _create_with_retry
print("✅ Rate-limit retry wrapper enabled")

✅ Groq client initialized
✅ Rate-limit retry wrapper enabled


## 📋 Section 4.5: Regenerate QA Set from Saved Docs

Regenerates evaluation Q&A pairs grounded in the **actual** documents in `data/raw/*.txt`
(so faithfulness, relevancy and refusal are all measurable and consistent with the KB).
The output is written to `qa_evaluation_set.txt`, which Section 14 later reads.

In [2]:
import json
import re
import time

QA_OUTPUT = "qa_evaluation_set.txt"

def load_raw_docs():
    """Load the three saved raw docs."""
    paths = {
        "product_terms": "data/raw/product_terms.txt",
        "fees":          "data/raw/fee_schedule.txt",
        "eligibility":   "data/raw/eligibility_rules.txt",
    }
    docs = {}
    for key, p in paths.items():
        with open(p, "r", encoding="utf-8", errors="replace") as f:
            docs[key] = f.read()
        print(f"\u2705 Loaded {key}: {len(docs[key])} chars")
    return docs

def parse_qa_response(raw: str) -> list:
    """Extract a JSON array from an LLM response, tolerating code fences."""
    raw = raw.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    raw = raw.strip()
    try:
        data = json.loads(raw)
        return data if isinstance(data, list) else []
    except json.JSONDecodeError:
        try:
            start = raw.index("[")
            end = raw.rindex("]") + 1
            data = json.loads(raw[start:end])
            return data if isinstance(data, list) else []
        except Exception:
            print("\u26a0\ufe0f JSON parse failed, returning empty list")
            return []

def generate_qa_for_doc(doc_key: str, doc_text: str, n: int = 10):
    """Generate n grounded Q&A pairs from one doc."""
    prompt = f"""
You are creating an evaluation dataset for a banking RAG system.

Read this {doc_key} document from the bank's knowledge base:
---
{doc_text[:4000]}
---

Create exactly {n} realistic customer questions that can be answered using ONLY this document.

Return ONLY a valid JSON array, no other text:
[
  {{"question": "...", "answer": "...", "answerable": true, "doc_type": "{doc_key}"}}
]
Requirements:
- Each question must be answerable and directly grounded in this document.
- Answers must cite exact values (amounts, rates, percentages, names) as written in the doc.
- Do NOT invent or infer facts that are not in the document.
"""
    print(f"\u23f3 Generating {n} Q&A pairs for {doc_key}...")
    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
        max_tokens=3000,
    )
    raw = response.choices[0].message.content
    pairs = parse_qa_response(raw)
    clean = []
    for p in pairs:
        if isinstance(p, dict) and p.get("question") and p.get("answer"):
            p["doc_type"] = doc_key
            p["answerable"] = bool(p.get("answerable", True))
            clean.append(p)
    print(f"\u2705 {doc_key}: {len(clean)} valid pairs")
    return clean

docs = load_raw_docs()

all_qa = []
for key, text in docs.items():
    all_qa.extend(generate_qa_for_doc(key, text, n=10))
    time.sleep(2)  # rate-limit buffer

with open(QA_OUTPUT, "w", encoding="utf-8") as f:
    f.write(f"QA EVALUATION SET ({len(all_qa)} Questions)\n\n")
    for i, q in enumerate(all_qa, start=1):
        f.write(f"{i}. Q: {q['question']} | A: {q['answer']}\n")

print(f"\n\u2705 Wrote {len(all_qa)} questions to {QA_OUTPUT}")


✅ Loaded product_terms: 9614 chars
✅ Loaded fees: 6742 chars
✅ Loaded eligibility: 10552 chars
⏳ Generating 10 Q&A pairs for product_terms...
✅ product_terms: 10 valid pairs
⏳ Generating 10 Q&A pairs for fees...
✅ fees: 10 valid pairs
⏳ Generating 10 Q&A pairs for eligibility...
✅ eligibility: 10 valid pairs

✅ Wrote 30 questions to qa_evaluation_set.txt


## 🔍 Section 5: Document Chunking & Ingestion

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from dataclasses import dataclass, field
from typing import List, Dict

@dataclass
class Chunk:
    chunk_id: str
    text: str
    doc_type: str   # 'product_terms' | 'fees' | 'eligibility'
    source_file: str

def load_and_chunk_documents() -> List[Chunk]:
    """Load raw docs and split into chunks with metadata."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=80,
        separators=["\n\n", "\n", ". ", " "]
    )

    doc_config = [
        ("data/raw/product_terms.txt", "product_terms"),
        ("data/raw/fee_schedule.txt", "fees"),
        ("data/raw/eligibility_rules.txt", "eligibility")
    ]

    all_chunks = []
    for filepath, doc_type in doc_config:
        with open(filepath, "r", encoding="utf-8") as f:
            text = f.read()

        splits = splitter.split_text(text)
        for i, chunk_text in enumerate(splits):
            all_chunks.append(Chunk(
                chunk_id=f"{doc_type}_{i:03d}",
                text=chunk_text.strip(),
                doc_type=doc_type,
                source_file=filepath
            ))
        print(f"✅ {doc_type}: {len(splits)} chunks")

    return all_chunks

chunks = load_and_chunk_documents()
print(f"\n📦 Total chunks: {len(chunks)}")

d:\ML projects\Vibecode\ai-news-agent\ai-news-agent\venv1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ product_terms: 33 chunks
✅ fees: 24 chunks
✅ eligibility: 33 chunks

📦 Total chunks: 90


## 🧠 Section 6: Build Dense Index (FAISS + BGE Embeddings)

In [4]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

print("⏳ Loading BGE embedding model...")
embed_model = SentenceTransformer("BAAI/bge-small-en-v1.5")
print("✅ Embedding model loaded")

def build_dense_index(chunks: List[Chunk]):
    """Encode chunks and build FAISS index."""
    texts = [c.text for c in chunks]
    print(f"⏳ Encoding {len(texts)} chunks...")

    # BGE requires a query prefix for retrieval
    embeddings = embed_model.encode(
        texts,
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True
    ).astype(np.float32)

    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)  # Inner product (cosine after normalization)
    index.add(embeddings)

    print(f"✅ FAISS index built: {index.ntotal} vectors, dim={dim}")
    return index, embeddings

faiss_index, chunk_embeddings = build_dense_index(chunks)

⏳ Loading BGE embedding model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3596.09it/s]


✅ Embedding model loaded
⏳ Encoding 90 chunks...


Batches: 100%|██████████| 3/3 [00:06<00:00,  2.01s/it]

✅ FAISS index built: 90 vectors, dim=384


## 📝 Section 7: Build Sparse Index (BM25)

In [5]:
from rank_bm25 import BM25Okapi
import re

def tokenize(text: str) -> List[str]:
    """Simple tokenizer for BM25."""
    text = text.lower()
    tokens = re.findall(r'\b[a-z0-9]+\b', text)
    return tokens

corpus_tokens = [tokenize(c.text) for c in chunks]
bm25 = BM25Okapi(corpus_tokens)

print(f"✅ BM25 index built over {len(corpus_tokens)} documents")

✅ BM25 index built over 90 documents


## 🔀 Section 8: Hybrid Retrieval with Reciprocal Rank Fusion (RRF)

In [6]:
from typing import Optional

def dense_search(query: str, top_k: int = 20, doc_type_filter: Optional[str] = None) -> List[tuple]:
    """Dense retrieval with optional doc_type filter. Returns (chunk_idx, score) list."""
    # BGE query prefix
    query_vec = embed_model.encode(
        [f"Represent this sentence for searching relevant passages: {query}"],
        normalize_embeddings=True
    ).astype(np.float32)

    scores, indices = faiss_index.search(query_vec, top_k * 3)  # over-fetch then filter
    results = []
    for idx, score in zip(indices[0], scores[0]):
        if idx == -1:
            continue
        if doc_type_filter and chunks[idx].doc_type != doc_type_filter:
            continue
        results.append((idx, float(score)))
        if len(results) == top_k:
            break
    return results


def sparse_search(query: str, top_k: int = 20, doc_type_filter: Optional[str] = None) -> List[tuple]:
    """BM25 retrieval with optional doc_type filter. Returns (chunk_idx, score) list."""
    query_tokens = tokenize(query)
    scores = bm25.get_scores(query_tokens)

    # Get ranked indices
    ranked_indices = np.argsort(scores)[::-1]
    results = []
    for idx in ranked_indices:
        if doc_type_filter and chunks[idx].doc_type != doc_type_filter:
            continue
        results.append((idx, float(scores[idx])))
        if len(results) == top_k:
            break
    return results


def reciprocal_rank_fusion(dense_results: List[tuple], sparse_results: List[tuple],
                           k: int = 60, top_n: int = 5) -> List[tuple]:
    """
    RRF: score(d) = sum(1 / (k + rank_i))
    Returns top_n (chunk_idx, rrf_score) pairs.
    """
    rrf_scores: Dict[int, float] = {}

    for rank, (idx, _) in enumerate(dense_results):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1 / (k + rank + 1)

    for rank, (idx, _) in enumerate(sparse_results):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1 / (k + rank + 1)

    sorted_results = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return sorted_results[:top_n]


def hybrid_retrieve(query: str, top_n: int = 5,
                    doc_type_filter: Optional[str] = None) -> List[Chunk]:
    """Full hybrid retrieval pipeline."""
    dense_res = dense_search(query, top_k=20, doc_type_filter=doc_type_filter)
    sparse_res = sparse_search(query, top_k=20, doc_type_filter=doc_type_filter)
    fused = reciprocal_rank_fusion(dense_res, sparse_res, top_n=top_n)
    return [chunks[idx] for idx, _ in fused], [score for _, score in fused]


# Quick test
test_chunks, test_scores = hybrid_retrieve("What is the minimum balance for savings account?")
print("\n🔍 Hybrid Retrieval Test:")
for chunk, score in zip(test_chunks, test_scores):
    print(f"  [{chunk.doc_type}] (rrf={score:.4f}) {chunk.text[:120]}...")


🔍 Hybrid Retrieval Test:
  [product_terms] (rrf=0.0325) | Feature | Details |
|---------|---------|
| **Base Interest Rate** | **3.00 % p.a.** on the first ₹2 Lakh of average d...
  [eligibility] (rrf=0.0308) | **Residency** | Resident Indian (NRI version exists separately). |
| **Existing Relationship** | Must hold at least on...
  [eligibility] (rrf=0.0303) ---

### Summary Table (Quick Reference)

| Product | Age | Income / Deposit | Minimum Balance / Installment | Credit Sc...
  [product_terms] (rrf=0.0299) | **Important T&C** | • Interest is posted monthly. <br>• Overdraft facility up to **₹1 Lakh** at 10.5 % p.a. (subject t...
  [fees] (rrf=0.0298) | Fee / Service | Standard Charge | Waiver / Discount Conditions |
|---------------|----------------|-------------------...


## 🎯 Section 9: Intent Router

Routes the query to the relevant document namespace before retrieval — reduces noise and improves precision.

In [7]:
INTENT_SYSTEM_PROMPT = """
You are an intent classifier for a banking assistant.
Classify the user query into ONE of these intents:

- fees: Questions about charges, penalties, costs, service fees, transaction fees
- eligibility: Questions about who can apply, requirements, documents, criteria, qualifications
- product_terms: Questions about features, interest rates, tenure, limits, how products work
- out_of_scope: Questions unrelated to banking products (stocks, insurance, crypto, general advice)

Respond with ONLY one word: fees | eligibility | product_terms | out_of_scope
"""

def classify_intent(query: str) -> str:
    """Classify query intent using LLM."""
    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {"role": "system", "content": INTENT_SYSTEM_PROMPT},
            {"role": "user", "content": query}
        ],
        temperature=0.0,
        max_tokens=10
    )
    intent = response.choices[0].message.content.strip().lower()
    valid_intents = {"fees", "eligibility", "product_terms", "out_of_scope"}
    return intent if intent in valid_intents else "product_terms"  # default fallback


# Test intent routing
test_queries = [
    "What is the NEFT charge for savings account?",
    "What is the minimum CIBIL score for personal loan?",
    "How does the fixed deposit interest work?",
    "Should I invest in Nifty 50 or Bitcoin?"
]

print("🎯 Intent Routing Tests:")
for q in test_queries:
    intent = classify_intent(q)
    print(f"  '{q[:60]}' → {intent}")
    time.sleep(0.5)

🎯 Intent Routing Tests:
  'What is the NEFT charge for savings account?' → product_terms
  'What is the minimum CIBIL score for personal loan?' → product_terms
  'How does the fixed deposit interest work?' → product_terms
  'Should I invest in Nifty 50 or Bitcoin?' → product_terms


## 🛡️ Section 10: Confidence Check & Refusal Mechanism

Two-layer refusal:
1. **Intent-level**: out_of_scope → immediate refusal
2. **Retrieval-level**: top RRF score below threshold → low-confidence refusal

In [8]:
CONFIDENCE_THRESHOLD = 0.007  # RRF score threshold (tune based on your data)

def check_retrieval_confidence(scores: List[float]) -> bool:
    """Returns True if retrieval is confident enough to answer."""
    if not scores:
        return False
    return scores[0] >= CONFIDENCE_THRESHOLD


REFUSAL_MESSAGE = (
    "I'm sorry, I don't have reliable information to answer this question. "
    "Please contact your branch or our 24/7 helpline at 1800-XXX-XXXX for assistance."
)

OUT_OF_SCOPE_MESSAGE = (
    "This question is outside the scope of NovaBank's product and fee information. "
    "I can only assist with questions about our savings accounts, fixed deposits, "
    "personal loans, credit cards, fees, and eligibility criteria."
)

print("✅ Refusal mechanism configured")
print(f"   Confidence threshold: {CONFIDENCE_THRESHOLD}")

✅ Refusal mechanism configured
   Confidence threshold: 0.007


## 💬 Section 11: Generator with Grounded Citations

In [9]:
GENERATOR_SYSTEM_PROMPT = """
You are a helpful and accurate banking assistant for NovaBank.
Answer the customer's question using ONLY the provided context chunks.

Rules:
1. Ground every factual claim in the context. Cite the source as [Source: chunk_id].
2. If the context doesn't contain enough information, say: 
   "Based on available information, I cannot fully answer this. Please contact our branch."
3. Be concise and professional. Use bullet points for lists of fees/features.
4. Do NOT make up numbers, rates, or policies not present in the context.
"""

def generate_answer(query: str, retrieved_chunks: List[Chunk]) -> str:
    """Generate cited answer from retrieved context."""
    context_str = "\n\n".join([
        f"[Source: {c.chunk_id}]\n{c.text}"
        for c in retrieved_chunks
    ])

    user_prompt = f"""Context:
{context_str}

Customer Question: {query}

Answer (cite sources as [Source: chunk_id]):"""

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {"role": "system", "content": GENERATOR_SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.1,
        max_tokens=600
    )
    return response.choices[0].message.content.strip()

print("✅ Generator configured")

✅ Generator configured


## 🔄 Section 12: Full RAG Pipeline

Putting it all together: Intent → Retrieve → Confidence Check → Generate

In [10]:
import time as time_module

def rag_pipeline(query: str, top_n: int = 5, verbose: bool = True) -> dict:
    """
    Full RAG pipeline.
    Returns: dict with answer, intent, retrieved_chunks, latency, refused flag
    """
    start = time_module.time()

    # Step 1: Intent Classification
    intent = classify_intent(query)
    if verbose:
        print(f"  🎯 Intent: {intent}")

    # Step 2: Out-of-scope refusal
    if intent == "out_of_scope":
        latency = time_module.time() - start
        return {
            "query": query,
            "answer": OUT_OF_SCOPE_MESSAGE,
            "intent": intent,
            "retrieved_chunks": [],
            "rrf_scores": [],
            "refused": True,
            "refusal_reason": "out_of_scope",
            "latency_ms": round(latency * 1000, 2)
        }

    # Step 3: Hybrid Retrieval (filtered by intent)
    retrieved, scores = hybrid_retrieve(query, top_n=top_n, doc_type_filter=intent)
    if verbose:
        top_score = scores[0] if scores else 0
        print(f"  🔍 Retrieved {len(retrieved)} chunks (top score: {top_score:.4f})")

    # Step 4: Confidence check
    if not check_retrieval_confidence(scores):
        latency = time_module.time() - start
        return {
            "query": query,
            "answer": REFUSAL_MESSAGE,
            "intent": intent,
            "retrieved_chunks": retrieved,
            "rrf_scores": scores,
            "refused": True,
            "refusal_reason": "low_confidence",
            "latency_ms": round((time_module.time() - start) * 1000, 2)
        }

    # Step 5: Generate answer with citations
    answer = generate_answer(query, retrieved)
    latency = time_module.time() - start

    return {
        "query": query,
        "answer": answer,
        "intent": intent,
        "retrieved_chunks": retrieved,
        "rrf_scores": scores,
        "refused": False,
        "refusal_reason": None,
        "latency_ms": round(latency * 1000, 2)
    }


# ---- DEMO RUNS ----
demo_queries = [
    "What are the NEFT charges for NovaSavings account?",
    # "What is the minimum CIBIL score required for NovaPersonal Loan?",
    # "Tell me about NovaFixed Deposit interest rates and tenure options.",
    "Should I buy gold or invest in mutual funds?"  # out of scope
]

for q in demo_queries:
    print(f"\n{'='*60}")
    print(f"❓ Query: {q}")
    result = rag_pipeline(q, verbose=True)
    print(f"  🛡️ Refused: {result['refused']} {('(' + result['refusal_reason'] + ')') if result['refused'] else ''}")
    print(f"  ⏱️ Latency: {result['latency_ms']}ms")
    print(f"  💬 Answer:\n{result['answer']}")
    time.sleep(1)


❓ Query: What are the NEFT charges for NovaSavings account?
  🎯 Intent: product_terms
  🔍 Retrieved 5 chunks (top score: 0.0328)
  🛡️ Refused: False 
  ⏱️ Latency: 1199.08ms
  💬 Answer:
Based on available information, I cannot fully answer this. Please contact our branch.

❓ Query: Should I buy gold or invest in mutual funds?
  🎯 Intent: product_terms
  🔍 Retrieved 5 chunks (top score: 0.0308)
  🛡️ Refused: False 
  ⏱️ Latency: 970.01ms
  💬 Answer:
Based on available information, I cannot fully answer this. Please contact our branch.


## ⚖️ Section 13: LLM-as-Judge Evaluator

Evaluates three metrics:
- **Faithfulness** — is the answer grounded in the retrieved context?
- **Answer Relevancy** — does the answer address the question?
- **Refusal Correctness** — did the system refuse when it should have (and not refuse when it shouldn't)?

In [11]:
FAITHFULNESS_PROMPT = """
You are evaluating whether an AI answer is faithful to the provided context.

Context:
{context}

Question: {question}
Answer: {answer}

Evaluate faithfulness: Does the answer ONLY contain claims supported by the context?
Score 0.0 to 1.0 where:
- 1.0 = all claims are grounded in context
- 0.5 = some claims unsupported
- 0.0 = answer contradicts or ignores context

Respond ONLY with a JSON object: {{"score": <float>, "reason": "<one sentence>"}}
"""

RELEVANCY_PROMPT = """
You are evaluating whether an AI answer addresses the user's question.

Question: {question}
Answer: {answer}

Score answer relevancy 0.0 to 1.0 where:
- 1.0 = directly and completely answers the question
- 0.5 = partially answers or goes off-topic
- 0.0 = completely misses the question

Respond ONLY with a JSON object: {{"score": <float>, "reason": "<one sentence>"}}
"""


def llm_judge(prompt_template: str, **kwargs) -> dict:
    """Run LLM judge with given prompt template."""
    prompt = prompt_template.format(**kwargs)
    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=150
    )
    raw = response.choices[0].message.content.strip()
    try:
        if raw.startswith("```"):
            raw = raw.split("```")[1]
            if raw.startswith("json"):
                raw = raw[4:]
        return json.loads(raw)
    except:
        return {"score": 0.0, "reason": "parse error"}


def evaluate_result(result: dict, ground_truth_answer: str = None,
                    should_be_refused: bool = False) -> dict:
    """Evaluate a single RAG result across all metrics."""
    metrics = {}

    # Refusal correctness
    if should_be_refused:
        metrics["refusal_correctness"] = 1.0 if result["refused"] else 0.0
    else:
        metrics["refusal_correctness"] = 0.0 if result["refused"] else 1.0

    if result["refused"]:
        metrics["faithfulness"] = None  # N/A for refused
        metrics["relevancy"] = None
        return metrics

    # Context for faithfulness
    context = "\n".join([c.text for c in result["retrieved_chunks"]])

    faith = llm_judge(FAITHFULNESS_PROMPT,
                      context=context,
                      question=result["query"],
                      answer=result["answer"])
    metrics["faithfulness"] = faith.get("score", 0.0)
    metrics["faithfulness_reason"] = faith.get("reason", "")

    time.sleep(0.5)

    rel = llm_judge(RELEVANCY_PROMPT,
                    question=result["query"],
                    answer=result["answer"])
    metrics["relevancy"] = rel.get("score", 0.0)
    metrics["relevancy_reason"] = rel.get("reason", "")

    return metrics


print("✅ LLM-as-Judge evaluator configured")

✅ LLM-as-Judge evaluator configured


## 📈 Section 14: Full Evaluation Run on QA Set

In [12]:
# Load QA set
import re

qa_set = []
with open("qa_evaluation_set.txt", "r", encoding="utf-8", errors="replace") as f:
    for line in f:
        line = line.strip()
        m = re.match(r"^(?:\d+\.\s*)?Q:\s*(.*?)\s*\|\s*A:\s*(.*)$", line)
        if m:
            qa_set.append({"question": m.group(1), "answer": m.group(2)})

# Run evaluation on a sample (run 10 queries for now)
EVAL_SAMPLE_SIZE = 10
eval_sample = qa_set[:EVAL_SAMPLE_SIZE]

print(f"\n📊 Running evaluation on {EVAL_SAMPLE_SIZE} samples...")
print("(This will take a few minutes due to API calls)\n")

eval_results = []
latencies = []

for i, qa in enumerate(eval_sample):
    print(f"  [{i+1}/{EVAL_SAMPLE_SIZE}] {qa['question'][:60]}...")

    # Run pipeline
    result = rag_pipeline(qa["question"], verbose=False)
    latencies.append(result["latency_ms"])

    # Evaluate
    should_refuse = not qa.get("answerable", True)
    metrics = evaluate_result(result, should_be_refused=should_refuse)

    eval_results.append({
        "question": qa["question"],
        "expected_answer": qa.get("answer", ""),
        "generated_answer": result["answer"],
        "intent": result["intent"],
        "refused": result["refused"],
        "should_refuse": should_refuse,
        "latency_ms": result["latency_ms"],
        **metrics
    })

    time.sleep(2)  # Rate limit buffer

print("\n✅ Evaluation complete!")


📊 Running evaluation on 10 samples...
(This will take a few minutes due to API calls)

  [1/10] What is the base interest rate for the NovaSavings Account o...
  [2/10] What fee is charged if I close my NovaSavings account within...
  [3/10] What is the maximum daily withdrawal limit for the NovaPrime...
  [4/10] What is the interest rate on the overdraft facility for the ...
  [5/10] What is the minimum balance requirement for an individual op...
  [6/10] What is the penalty for premature withdrawal of a fixed depo...
  [7/10] What is the interest rate for a fixed deposit with a tenure ...
  [8/10] What is the annual fee for the NovaPrime metal debit card an...
  [9/10] What monthly credit amount is required to waive the minimum ...
  [10/10] What is the maximum overdraft facility available for a NovaP...

✅ Evaluation complete!


## 📊 Section 15: Metrics Summary

In [13]:
import statistics

# Filter answered vs refused
answered = [r for r in eval_results if not r["refused"]]
refused_results = [r for r in eval_results if r["refused"]]

# Faithfulness
faith_scores = [r["faithfulness"] for r in answered if r.get("faithfulness") is not None]
avg_faithfulness = statistics.mean(faith_scores) if faith_scores else 0

# Relevancy
rel_scores = [r["relevancy"] for r in answered if r.get("relevancy") is not None]
avg_relevancy = statistics.mean(rel_scores) if rel_scores else 0

# Refusal correctness
refusal_correct = [r for r in eval_results if r["refusal_correctness"] == 1.0]
refusal_correctness = len(refusal_correct) / len(eval_results) if eval_results else 0

# Latency
sorted_latencies = sorted(latencies)
p95_latency = sorted_latencies[int(len(sorted_latencies) * 0.95)] if sorted_latencies else 0
avg_latency = statistics.mean(latencies) if latencies else 0

print("\n" + "="*50)
print("📊 EVALUATION RESULTS")
print("="*50)
print(f"  Samples evaluated    : {len(eval_results)}")
print(f"  Answered             : {len(answered)}")
print(f"  Refused              : {len(refused_results)}")
print(f"")
print(f"  Faithfulness         : {avg_faithfulness:.3f}")
print(f"  Answer Relevancy     : {avg_relevancy:.3f}")
print(f"  Refusal Correctness  : {refusal_correctness:.3f}")
print(f"")
print(f"  Avg Latency          : {avg_latency:.0f}ms")
print(f"  p95 Latency          : {p95_latency:.0f}ms")
print("="*50)

# Save results
with open("eval_results.json", "w") as f:
    json.dump(eval_results, f, indent=2, default=str)

summary = {
    "samples": len(eval_results),
    "faithfulness": round(avg_faithfulness, 3),
    "answer_relevancy": round(avg_relevancy, 3),
    "refusal_correctness": round(refusal_correctness, 3),
    "avg_latency_ms": round(avg_latency, 1),
    "p95_latency_ms": round(p95_latency, 1)
}
with open("summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\n✅ Results saved to data/eval/")


📊 EVALUATION RESULTS
  Samples evaluated    : 10
  Answered             : 10
  Refused              : 0

  Faithfulness         : 0.400
  Answer Relevancy     : 0.900
  Refusal Correctness  : 1.000

  Avg Latency          : 6792ms
  p95 Latency          : 11237ms

✅ Results saved to data/eval/


## 🔬 Section 16: Error Analysis

In [14]:
# Low faithfulness cases
low_faith = [
    r for r in answered
    if r.get("faithfulness") is not None and r["faithfulness"] < 0.7
]

# Wrong refusals
wrong_refusals = [r for r in eval_results if r["refusal_correctness"] == 0.0]

print(f"⚠️  Low Faithfulness Cases (< 0.7): {len(low_faith)}")
for r in low_faith[:3]:
    print(f"  Q: {r['question'][:80]}")
    print(f"  Faith: {r['faithfulness']:.2f} | {r.get('faithfulness_reason', '')}")
    print()

print(f"\n⚠️  Wrong Refusal Decisions: {len(wrong_refusals)}")
for r in wrong_refusals[:3]:
    refused_when = "should NOT have" if not r["should_refuse"] else "should have"
    print(f"  Q: {r['question'][:80]}")
    print(f"  System refused={r['refused']} but {refused_when} refused")
    print()

⚠️  Low Faithfulness Cases (< 0.7): 6
  Q: What is the base interest rate for the NovaSavings Account on the first ₹2 lakh 
  Faith: 0.00 | parse error

  Q: What is the interest rate on the overdraft facility for the NovaSavings Account?
  Faith: 0.00 | parse error

  Q: What is the penalty for premature withdrawal of a fixed deposit with a tenure of
  Faith: 0.00 | parse error


⚠️  Wrong Refusal Decisions: 0


## 🎮 Section 17: Interactive Demo

In [ ]:
def interactive_query(query: str):
    """Pretty-print a single query through the full pipeline."""
    print("\n" + "="*60)
    print(f"❓ Query: {query}")
    print("-"*60)

    result = rag_pipeline(query, verbose=True)

    print(f"\n💬 Answer:")
    print(result["answer"])

    if result["retrieved_chunks"] and not result["refused"]:
        print(f"\n📎 Sources used:")
        for chunk in result["retrieved_chunks"]:
            print(f"  • [{chunk.chunk_id}] ({chunk.doc_type}): {chunk.text[:80]}...")

    print(f"\n⏱️ Latency: {result['latency_ms']}ms")
    print("="*60)


# Try your own queries!
interactive_query("What documents do I need to apply for a NovaPersonal Loan?")

In [ ]:
# Try more queries
interactive_query("What is the annual fee for NovaCreditCard Platinum?")

In [ ]:
interactive_query("How do I invest in the stock market?")  # Should be refused